## Import necessary modules
Run this cell before running any other cells

In [ ]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np

LOG.propagate = False

# Printing and Logging
## Printing
You can use the **print()** function in Python to print messages to the screen. <br>
The message can be a string, or any other object, the object will be converted into a string before it is written to the screen. <br>

## Logging
You could use the logging module that is setup in *utils.py*. <br>
It prints to both your screen (standard output) as well as to log files (*ble.log*) in the *logs* directory. <br>
This is the recommended way to output messages, since the log files can help with debugging. <br>
The logging module also provides different log levels as shown below, each formatted with a different color for increased visibility. <br>

__**NOTE**__: You may notice that the DEBUG message is not printed to the screen but is printed in the log file. This is because the logging level for the screen is set to INFO and for the file is set to DEBUG. You can change the default log levels in *utils.py* (**STREAM_LOG_LEVEL** and **FILE_LOG_LEVEL**). 

## Formatting output
To format your strings, you may use %-formatting, str.format() or f-strings. <br>
The most "pythonic" way would be to use f-strings. [Here](https://realpython.com/python-f-strings/) is a good tutorial on f-strings. <br>

In [ ]:
LOG.debug("debug")
LOG.info("info")
LOG.warning("warning")
LOG.error("error")
LOG.critical("critical")

<hr>

# BLE
## ArtemisBLEController
The class **ArtemisBLEController** (defined in *ble.py*) provides member functions to handle various BLE operations to send and receive data to/from the Artemis board, provided the accompanying Arduino sketch is running on the Artemis board. <br>

<table align="left">
     <tr>
        <th style="text-align: left; font-size: medium">Member Functions</th>
        <th style="text-align: left; font-size: medium">Description</th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">reload_config()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Reload changes made in <em>connection.yaml.</em></span></th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">connect()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Connect to the Artemis board, whose MAC address is specified in <em>connection.yaml</em>.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">disconnect()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Disconnect from the Artemis board.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">is_connected()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Return a boolean indicating whether your controller is connected to the Artemis board or not.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">send_command(cmd_type, data)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Send the command <strong>cmd_type</strong> (integer) with <strong>data</strong> (string) to the Artemis board.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">receive_float(uuid) <br> receive_string(uuid) <br> receive_int(uuid)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Read the GATT characteristic (specified by its <strong>uuid</strong>) of type float, string or int. <br> The type of the GATT
            characteristic is determined by the classes BLEFloatCharacteristic, BLECStringCharacteristic or
            BLEIntCharacteristic in the Arduino sketch.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">start_notify(uuid, notification_handler)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Activate notifications on the GATT characteristic (specified by its <strong>uuid</strong>). <br> <strong>notification_handler</strong> is a
            function callback which must accept two inputs; the first will be a uuid string object and the second will
            be the bytearray of the characteristic value.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">bytearray_to_float(byte_array) <br> bytearray_to_string(byte_array) <br> bytearray_to_int(byte_array)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Convert the <strong>bytearray</strong> to float, string or int, respectively. <br> You may use these functions inside your
            notification callback function.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">stop_notify(uuid)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Stop notifications on the GATT characteristic (specified by its <strong>uuid</strong>).</span></th>
    </tr>
</table>

<table align="left">
     <tr>
        <th style="text-align: left; font-size: medium">Member Variables</th>
        <th style="text-align: left; font-size: medium">Description</th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">uuid</span></th>
        <th style="text-align: left"><span style="font-weight: normal">A dictionary that stores the UUIDs of the various characteristics specified in <em>connection.yaml</em>.</span></th>
    </tr>
</table>

## Configuration
- The MAC address, Service UUID and GATT characteristic UUIDs are defined in the file: *connection.yaml*.
- They should match the UUIDs used in the Arduino sketch.
- The artemis board running the base code should display its MAC address in the serial monitor.
- Update the **artemis_address** in *connection.yaml*, accordingly.
- Make sure to call **ble.reload_config()** or **get_ble_controller()** (which internally calls **reload_config()**) after making any changes to your configuration file.

<hr>

In the below cell, we create an **ArtemisBLEController** object using **get_ble_controller()** (defined in *ble.py*), which creates and/or returns a single instance of **ArtemisBLEController**. <br>
<span style="color:rgb(240,50,50)"> __NOTE__: Do not use the class directly to instantiate an object. </span><br>

In [ ]:
# Get ArtemisBLEController object
ble = get_ble_controller()

# Connect to the Artemis Device
ble.connect()

In [ ]:
ble.reload_config()

## Receive data from the Artemis board

The cell below shows examples of reading different types (as defined in the Arduino sketch) of GATT characteristics.

In [ ]:
# Read a float GATT Charactersistic
f = ble.receive_float(ble.uuid['RX_FLOAT'])
print(f)

In [ ]:
# Read a string GATT Charactersistic
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

## Send a command to the Artemis board
Send the PING command and read the reply string from the string characteristic RX_STRING. <br>
__NOTE__: The **send_command()** essentially sends a string data to the GATT characteristic (TX_CMD_STRING). The GATT characteristic in the Arduino sketch is of type BLECStringCharacteristic.

In [ ]:
ble.send_command(CMD.PING, "")

In [ ]:
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

The cell below shows an example of the SEND_TWO_INTS command. <br> The two values in the **data** are separated by a delimiter "|". <br>
Refer Lab 2 documentation for more information on the command protocol.

In [ ]:
ble.send_command(CMD.SEND_TWO_INTS, "2|-6")

The Artemis board should print the two integers to the serial monitor in the ArduinoIDE. 

## Disconnect

In [ ]:
# Disconnect
ble.disconnect()

## Task 1
Send a string value from the computer to the Artemis board using the ECHO command. The computer should then receive and print an augmented string.

In [ ]:
ble.send_command(CMD.ECHO, "HiHello")

## Task 2
Send three floats to the Artemis board using the SEND_THREE_FLOATS command and extract the three float values in the Arduino sketch.

In [ ]:
ble.send_command(CMD.SEND_THREE_FLOATS, "0.1|0.2|0.25")

## Task 3
Add a command GET_TIME_MILLIS which makes the robot reply write a string such as “T:123456” to the string characteristic.

In [ ]:
ble.send_command(CMD.GET_TIME_MILLIS,"")

In [ ]:
ble.receive_string(ble.uuid["RX_STRING"])

## Task 4
Setup a notification handler in Python to receive the string value (the BLEStringCharactersitic in Arduino) from the Artemis board. In the callback function, extract the time from the string.

In [ ]:
timestamps = []
def time_notification_handler(sender, data: bytearray):
    s = ble.bytearray_to_string(data)
    if s.startswith("T:"):
        t = int(s[2:])
        timestamps.append(t)
        print("Time(ms) =", t)

In [ ]:
ble.start_notify(ble.uuid["RX_STRING"], time_notification_handler)

In [ ]:
ble.send_command(CMD.GET_TIME_MILLIS, "")

In [ ]:
ble.stop_notify(ble.uuid["RX_STRING"])

## Task 5
Write a loop that gets the current time in milliseconds and sends it to your laptop to be received and processed by the notification handler. Collect these values for a few seconds and use the time stamps to determine how fast messages can be sent. What is the effective data transfer rate of this method?

In [ ]:
timestamps = []
duration = 2.0
start = time.time()

while time.time() - start < duration:
    ble.send_command(CMD.GET_TIME_MILLIS, "")

In [ ]:
rate = (len(timestamps))/ duration
print ("Transfer rate (msg/s)", rate)
print ("Sending Time (s/msg)", 1/rate)

## Task 6
Now create an array that can store time stamps. This array should be defined globally so that other functions can access it if need be. In the loop, rather than send each time stamp, place each time stamp into the array. (Note: you’ll need some extra logic to determine when your array is full so you don’t “over fill” the array.) Then add a command SEND_TIME_DATA which loops the array and sends each data point as a string to your laptop to be processed. (You can store these values in a list in python to determine if all the data was sent over.)

In [ ]:
timestamps = []
ble.send_command(CMD.START_COLLECT_DATA, "")
time.sleep(duration)
ble.send_command(CMD.SEND_TIME_DATA, "")

In [ ]:
rate = (len(timestamps))/ duration
print ("Transfer rate (msg/s)", rate)
print ("Sending Time (s/msg)", 1/rate)

## ICM Sensor

In [ ]:
times = []
accX = []
accY = []
accZ = []

def accl_notification_handler(sender, data: bytearray):
    s = ble.bytearray_to_string(data).split("|")
    if len(s) == 3:
        x = float(s[0][2:])
        y = float(s[1][2:])
        z = float(s[2][2:])
        accX.append(x)
        accY.append(y)
        accZ.append(z)
        print ("X:", x, ", Y:", y, "Z:", z)

In [ ]:
ble.start_notify(ble.uuid["RX_STRING"], accl_notification_handler)

In [ ]:
times = []
accX = []
accY = []
accZ = []

ble.send_command(CMD.START_COLLECT_DATA, "")
time.sleep(2)
ble.send_command(CMD.GET_ACCL_READINGS, "")

In [ ]:
accX = np.array(accX)
accY = np.array(accY)
accZ = np.array(accZ)

pitch_deg = np.degrees(np.arctan2(accX, np.sqrt(accY**2 + accZ**2)))
roll_deg  = np.degrees(np.arctan2(accY, np.sqrt(accX**2 + accZ**2)))

In [ ]:
for i, (p, r) in enumerate(zip(pitch_deg, roll_deg)):
    print(f"pitch={p:7.2f}°   roll={r:7.2f}°")

## Task 7
Add a second array that is the same size as the time stamp array. Use this array to store temperature readings. Each element in both arrays should correspond, e.e., the first time stamp was recorded at the same time as the first temperature reading. Then add a command GET_TEMP_READINGS that loops through both arrays concurrently and sends each temperature reading with a time stamp. The notification handler should parse these strings and add populate the data into two lists.

In [ ]:
times = []
temps = []

def temp_notification_handler(sender, data: bytearray):
    s = ble.bytearray_to_string(data).split("|")
    if len(s) == 2:
        time = int(s[0][2:])
        temp = float(s[1][2:])
        times.append(time)
        temps.append(temp)
        print ("T:", time, ", F:", temp)

In [ ]:
ble.start_notify(ble.uuid["RX_STRING"], temp_notification_handler)

In [ ]:
times = []
temps = []

ble.send_command(CMD.START_COLLECT_DATA, "")
time.sleep(duration)
ble.send_command(CMD.GET_TEMP_READINGS, "")

In [ ]:
ble.stop_notify(ble.uuid["RX_STRING"])